
<div style="background:#07111F; padding:28px; border-radius:18px; border:1px solid rgba(255,255,255,0.14);">
  <h1 style="color:#E6EDF3; margin:0; font-size:34px; text-align:center;">
    Multidimensional Energy and Operability Visualization Lab
  </h1>
  <p style="color:#C9D1D9; margin:10px 0 0 0; font-size:14px; text-align:center;">
    3D, 4D, interactive, and fly-through visuals for data center energy, weather, and interoperability analytics
  </p>
  <p style="color:#C9D1D9; margin:6px 0 0 0; font-size:13px; text-align:center; opacity:0.92;">
    By DaScient
  </p>
</div>



## Scope

This notebook is an educational and operational visualization package.

It provides:

- interactive, high-fidelity visual analytics for multi-site fleets
- 3D and 4D views (time as animation / frames)
- color-gradient surfaces and risk fields
- fly-through camera paths for executive storytelling
- actionable drilldowns: drivers, clusters, and mitigation recommendations

Data sources

- Runs end to end using synthetic data by default.
- Includes adapters to ingest your existing exports (CSV) from prior DaScient notebooks.

Outputs

- interactive figures (Plotly)
- exportable HTML snapshots for sharing and embedding



## Contents

1. Setup  
2. Data generation and ingestion adapters  
3. Multi-dimensional metric cube  
4. 3D scatter: energy vs carbon vs SLO risk  
5. 3D surface: weather-to-energy field  
6. 4D animation: time-evolving fleet behavior  
7. Parallel coordinates: multi-metric tradeoffs  
8. Fly-through: guided camera tour of risk space  
9. Operational views: mitigation decision panels  
10. Export: HTML and JSON artifacts  



## 1. Setup

This notebook uses Plotly for interactive 3D and animated visuals.

If you are in a clean environment, run the install cell once.


In [ ]:

# Optional installs. Uncomment if needed.
# %pip install -q numpy pandas scipy plotly ipywidgets kaleido


In [ ]:

import os, json, time, math, random, hashlib
from dataclasses import dataclass
from typing import Dict, Any, Tuple

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

def utc_now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def set_seed(seed: int = 7) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def stable_hash(obj: Any) -> str:
    blob = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]

RUN_TS = utc_now_iso()
set_seed(7)

RUN_TS



## 2. Data generation and ingestion adapters

Expected minimal schema

- time_utc
- site, region, workload
- it_kw, facility_kw
- carbon_g_per_kwh
- rps, p95_ms, error_rate
- throttling_rate, power_capping_rate
- telemetry_coverage, api_drift_events, migration_events, migration_p95_delta_ms

Replace the synthetic generator by reading your CSV and mapping columns.


In [ ]:

@dataclass
class FleetConfig:
    n_days: int = 21
    freq_min: int = 30
    sites: Tuple[str, ...] = ("SLC-1", "IAD-2", "DFW-1", "PDX-1")
    regions: Tuple[str, ...] = ("RockyMountain", "PJM", "ERCOT", "PNW")
    workloads: Tuple[str, ...] = ("Search", "Ads", "Training", "Inference", "Storage")
    seed: int = 7

cfg = FleetConfig()

def make_time_index(n_days: int, freq_min: int) -> pd.DatetimeIndex:
    start = pd.Timestamp.utcnow().floor("D") - pd.Timedelta(days=n_days)
    return pd.date_range(start=start, periods=int(n_days * 24 * 60 / freq_min), freq=f"{freq_min}min", tz="UTC")

def diurnal_profile(ts: pd.DatetimeIndex) -> np.ndarray:
    hour = ts.hour + ts.minute / 60.0
    return 0.65 + 0.35 * np.sin((hour - 7) / 24.0 * 2 * np.pi)

def generate_fleet(cfg: FleetConfig) -> pd.DataFrame:
    set_seed(cfg.seed)
    ts = make_time_index(cfg.n_days, cfg.freq_min)
    diurnal = diurnal_profile(ts)

    site_params = {
        "SLC-1": {"base_pue": 1.22, "pue_var": 0.06, "grid_gpkwh": 420},
        "IAD-2": {"base_pue": 1.30, "pue_var": 0.07, "grid_gpkwh": 350},
        "DFW-1": {"base_pue": 1.35, "pue_var": 0.09, "grid_gpkwh": 520},
        "PDX-1": {"base_pue": 1.18, "pue_var": 0.05, "grid_gpkwh": 160},
    }

    workload_params = {
        "Search": {"rps_per_it_kw": 38, "p95_base": 46, "err_base": 0.004, "gpu_frac": 0.05},
        "Ads": {"rps_per_it_kw": 32, "p95_base": 55, "err_base": 0.006, "gpu_frac": 0.10},
        "Training": {"rps_per_it_kw": 0.15, "p95_base": 0,  "err_base": 0.002, "gpu_frac": 0.80},
        "Inference": {"rps_per_it_kw": 11, "p95_base": 85, "err_base": 0.008, "gpu_frac": 0.55},
        "Storage": {"rps_per_it_kw": 5, "p95_base": 20, "err_base": 0.003, "gpu_frac": 0.00},
    }

    rows = []
    for site, region in zip(cfg.sites, cfg.regions):
        sp = site_params[site]
        base_it_kw = {"SLC-1": 8200, "IAD-2": 9800, "DFW-1": 7600, "PDX-1": 6900}[site]

        for wl in cfg.workloads:
            wp = workload_params[wl]
            wl_share = {"Search": 0.28, "Ads": 0.18, "Training": 0.20, "Inference": 0.22, "Storage": 0.12}[wl]

            it_kw = base_it_kw * wl_share * (0.75 + 0.45 * diurnal) * (1 + 0.06 * np.random.randn(len(ts)))
            it_kw = np.clip(it_kw, 0, None)

            pue = sp["base_pue"] + sp["pue_var"] * np.sin(np.linspace(0, 5*np.pi, len(ts))) + 0.015*np.random.randn(len(ts))
            pue = np.clip(pue, 1.05, 1.8)

            facility_kw = it_kw * pue
            carbon = sp["grid_gpkwh"] * (0.85 + 0.25 * (1 - diurnal)) * (0.95 + 0.08*np.random.randn(len(ts)))
            carbon = np.clip(carbon, 40, 900)

            cpu_util = np.clip(0.35 + 0.45*(it_kw / (base_it_kw * wl_share + 1e-9)) + 0.08*np.random.randn(len(ts)), 0, 1)
            gpu_util = np.clip(wp["gpu_frac"] * (0.55 + 0.55*(it_kw / (base_it_kw * wl_share + 1e-9))) + 0.12*np.random.randn(len(ts)), 0, 1)

            temp_c = np.clip(22 + 10*cpu_util + 6*gpu_util + 3*(pue - sp["base_pue"]) + 1.2*np.random.randn(len(ts)), 16, 45)
            throttling = np.clip((temp_c - 32) / 14, 0, 1) * (0.35 + 0.65*gpu_util)

            power_cap = np.clip((facility_kw - np.quantile(facility_kw, 0.92)) / (np.quantile(facility_kw, 0.98) - np.quantile(facility_kw, 0.92) + 1e-9), 0, 1)
            power_cap = np.clip(power_cap + 0.06*np.random.randn(len(ts)), 0, 1)

            rps = np.clip(wp["rps_per_it_kw"] * it_kw * (1 - 0.12*throttling) * (0.95 + 0.06*np.random.randn(len(ts))), 0, None)

            p95 = wp["p95_base"] + 55*throttling + 30*power_cap + 12*np.maximum(cpu_util - 0.85, 0) + 4*np.random.randn(len(ts))
            p95 = np.clip(p95, 1, None) if wp["p95_base"] > 0 else np.zeros(len(ts))

            err = np.clip(wp["err_base"] + 0.020*throttling + 0.012*power_cap + 0.010*np.maximum(cpu_util - 0.90, 0) + 0.002*np.random.rand(len(ts)), 0, 0.25)

            telemetry_coverage = np.clip(0.88 + 0.10*np.random.randn(len(ts)) - 0.12*(wl == "Training") + 0.05*(site == "PDX-1"), 0.3, 1.0)
            api_drift = np.random.poisson(lam=0.05 + 0.10*(wl == "Inference") + 0.06*(site == "DFW-1"), size=len(ts))

            migrations = np.random.poisson(lam=0.02 + 0.03*(wl in ["Inference","Search"]) + 0.02*(site == "IAD-2"), size=len(ts))
            mig_p95_delta = np.where(migrations > 0, np.clip(15 + 25*np.random.randn(len(ts)), -10, 140), 0.0)

            for i in range(len(ts)):
                rows.append({
                    "time_utc": ts[i],
                    "site": site,
                    "region": region,
                    "workload": wl,
                    "it_kw": float(it_kw[i]),
                    "facility_kw": float(facility_kw[i]),
                    "carbon_g_per_kwh": float(carbon[i]),
                    "rps": float(rps[i]),
                    "p95_ms": float(p95[i]),
                    "error_rate": float(err[i]),
                    "cpu_util": float(cpu_util[i]),
                    "gpu_util": float(gpu_util[i]),
                    "temp_c": float(temp_c[i]),
                    "throttling_rate": float(throttling[i]),
                    "power_capping_rate": float(power_cap[i]),
                    "telemetry_coverage": float(telemetry_coverage[i]),
                    "api_drift_events": int(api_drift[i]),
                    "migration_events": int(migrations[i]),
                    "migration_p95_delta_ms": float(mig_p95_delta[i]),
                })

    return pd.DataFrame(rows)

df = generate_fleet(cfg)
df.head(), df.shape



## 3. Multi-dimensional metric cube

Derived metrics:

- PUE and energy
- CO2e per request
- Joules per request
- SLO compliance and a smooth SLO risk proxy


In [ ]:

def add_derived_metrics(df: pd.DataFrame, freq_min: int) -> pd.DataFrame:
    out = df.copy()
    hours = freq_min / 60.0

    out["pue"] = out["facility_kw"] / np.maximum(out["it_kw"], 1e-9)
    out["facility_kwh"] = out["facility_kw"] * hours
    out["requests"] = out["rps"] * (hours * 3600.0)

    out["co2e_kg"] = out["facility_kwh"] * (out["carbon_g_per_kwh"] / 1000.0)
    out["co2e_g_per_request"] = (out["co2e_kg"] * 1000.0) / np.maximum(out["requests"], 1.0)

    out["joules"] = out["facility_kwh"] * 3.6e6
    out["joules_per_request"] = out["joules"] / np.maximum(out["requests"], 1.0)

    SLO_P95_MS = 120.0
    SLO_ERR = 0.01
    out["slo_ok"] = ((out["p95_ms"] <= SLO_P95_MS) | (out["p95_ms"] == 0.0)) & (out["error_rate"] <= SLO_ERR)

    out["slo_risk"] = (
        1.6*np.maximum((out["p95_ms"] - SLO_P95_MS) / SLO_P95_MS, 0.0)
        + 1.4*np.maximum((out["error_rate"] - SLO_ERR) / SLO_ERR, 0.0)
        + 0.9*out["throttling_rate"]
        + 0.8*out["power_capping_rate"]
        + 0.7*np.maximum(0.85 - out["telemetry_coverage"], 0.0)
    )
    out["slo_risk"] = np.clip(out["slo_risk"], 0, 6.0)
    return out

dfm = add_derived_metrics(df, cfg.freq_min)
dfm[["pue","co2e_g_per_request","joules_per_request","slo_risk"]].describe().T



## 4. 3D scatter: energy vs carbon vs risk


In [ ]:

sample = dfm.sample(n=min(12000, len(dfm)), random_state=7)

fig = px.scatter_3d(
    sample,
    x="joules_per_request",
    y="co2e_g_per_request",
    z="slo_risk",
    color="throttling_rate",
    size="facility_kw",
    hover_data=["site","workload","pue","p95_ms","error_rate","power_capping_rate","telemetry_coverage"],
    opacity=0.65,
    title="3D risk space: Joules per request vs CO2e per request vs SLO risk"
)
fig.update_layout(height=720)
fig.show()



## 5. 3D surface: weather-to-energy field (synthetic)


In [ ]:

temp = np.linspace(-5, 45, 90)
rh = np.linspace(10, 100, 90)
T, RH = np.meshgrid(temp, rh)

cool = np.maximum((T - 18)/22, 0.0)
humid = np.maximum((RH - 60)/40, 0.0)
Z = 1.0 + 0.75*(cool**1.6) + 0.35*(humid**1.4) + 0.22*(cool*humid)

fig = go.Figure(data=[go.Surface(x=T, y=RH, z=Z, colorscale="Viridis")])
fig.update_layout(
    title="Cooling amplification surface (planning proxy)",
    scene=dict(
        xaxis_title="Temperature (C)",
        yaxis_title="Relative humidity (%)",
        zaxis_title="Cooling multiplier"
    ),
    height=720
)
fig.show()



## 6. 4D animation: time-evolving fleet behavior


In [ ]:

dfm["date"] = dfm["time_utc"].dt.floor("D")

site_day = dfm.groupby(["date","site"], as_index=False).agg(
    facility_kw=("facility_kw","sum"),
    co2e_g_per_request=("co2e_g_per_request","median"),
    slo_risk=("slo_risk","median"),
    throttling_rate=("throttling_rate","median"),
    pue=("pue","median"),
)

fig = px.scatter_3d(
    site_day,
    x="facility_kw",
    y="co2e_g_per_request",
    z="slo_risk",
    color="site",
    size="facility_kw",
    animation_frame=site_day["date"].astype(str),
    hover_data=["pue","throttling_rate"],
    opacity=0.75,
    title="4D animation: site behavior over time"
)
fig.update_layout(height=760)
fig.show()



## 7. Parallel coordinates: multi-metric tradeoffs


In [ ]:

pc = dfm.sample(n=min(8000, len(dfm)), random_state=9).copy()

fig = px.parallel_coordinates(
    pc,
    dimensions=[
        "pue",
        "throttling_rate",
        "power_capping_rate",
        "telemetry_coverage",
        "joules_per_request",
        "co2e_g_per_request",
        "slo_risk",
    ],
    color="slo_risk",
    color_continuous_scale="Turbo",
    title="Parallel coordinates: multi-metric tradeoffs"
)
fig.update_layout(height=720)
fig.show()



## 8. Fly-through: guided camera tour of risk space


In [ ]:

s2 = dfm.sample(n=min(6000, len(dfm)), random_state=21)

trace = go.Scatter3d(
    x=s2["joules_per_request"],
    y=s2["co2e_g_per_request"],
    z=s2["slo_risk"],
    mode="markers",
    marker=dict(
        size=3.6,
        color=s2["slo_risk"],
        colorscale="Plasma",
        opacity=0.75,
        colorbar=dict(title="slo_risk")
    ),
    text=s2["site"] + " / " + s2["workload"],
    hovertemplate=" %{text}<br>J/request=%{x:.0f}<br>g/request=%{y:.2f}<br>risk=%{z:.2f}<extra></extra>",
)

def cam(eye_x, eye_y, eye_z):
    return dict(eye=dict(x=float(eye_x), y=float(eye_y), z=float(eye_z)))

frames = []
steps = 70
for i in range(steps):
    ang = 2*math.pi * (i/steps)
    r = 2.2 - 1.2*(i/steps)
    eye = cam(r*math.cos(ang), r*math.sin(ang), 1.2 + 0.7*math.sin(ang*0.5))
    frames.append(go.Frame(layout=go.Layout(scene_camera=eye), name=str(i)))

fig = go.Figure(data=[trace], frames=frames)
fig.update_layout(
    title="Fly-through: guided camera tour of risk space",
    scene=dict(
        xaxis_title="Joules per request",
        yaxis_title="CO2e g per request",
        zaxis_title="SLO risk",
    ),
    height=760,
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "buttons": [
            {"label": "Play", "method": "animate",
             "args": [None, {"frame": {"duration": 60, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}]},
            {"label": "Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
        ],
    }],
)
fig.show()



## 9. Operational views: mitigation decision panels


In [ ]:

dfm["driver_throttle"] = 0.9*dfm["throttling_rate"]
dfm["driver_cap"] = 0.8*dfm["power_capping_rate"]
dfm["driver_telemetry"] = 0.7*np.maximum(0.85 - dfm["telemetry_coverage"], 0.0)
dfm["driver_slo"] = dfm["slo_risk"] - (dfm["driver_throttle"] + dfm["driver_cap"] + dfm["driver_telemetry"])
dfm["driver_slo"] = np.maximum(dfm["driver_slo"], 0.0)

panel = dfm.groupby(["site","workload"], as_index=False).agg(
    risk=("slo_risk","median"),
    pue=("pue","median"),
    joules_per_request=("joules_per_request","median"),
    co2e_g_per_request=("co2e_g_per_request","median"),
    throttling_rate=("throttling_rate","median"),
    power_capping_rate=("power_capping_rate","median"),
    telemetry_coverage=("telemetry_coverage","median"),
    d_throttle=("driver_throttle","median"),
    d_cap=("driver_cap","median"),
    d_telemetry=("driver_telemetry","median"),
    d_slo=("driver_slo","median"),
)

def recommend(row) -> str:
    drivers = {"thermal/throttle": row["d_throttle"], "power/cap": row["d_cap"], "observability": row["d_telemetry"], "slo/regression": row["d_slo"]}
    top = max(drivers, key=drivers.get)
    if top == "thermal/throttle":
        return "Thermal protocol: rebalance placement, thermal-aware scheduling, cooling checks"
    if top == "power/cap":
        return "Power protocol: power budgets, admission control, shift batch"
    if top == "observability":
        return "Interop protocol: instrumentation standardization, minimum coverage gate"
    return "SLO protocol: canary regressions, release gate, rollback policy"

panel["recommendation"] = panel.apply(recommend, axis=1)
panel = panel.sort_values("risk", ascending=False).reset_index(drop=True)
panel.head(20)


In [ ]:

top = panel.head(18).copy()
fig = go.Figure()
fig.add_trace(go.Bar(name="thermal/throttle", x=top["site"]+" / "+top["workload"], y=top["d_throttle"]))
fig.add_trace(go.Bar(name="power/cap", x=top["site"]+" / "+top["workload"], y=top["d_cap"]))
fig.add_trace(go.Bar(name="observability", x=top["site"]+" / "+top["workload"], y=top["d_telemetry"]))
fig.add_trace(go.Bar(name="slo/regression", x=top["site"]+" / "+top["workload"], y=top["d_slo"]))
fig.update_layout(
    barmode="stack",
    title="Top risk clusters with driver decomposition",
    xaxis_title="site / workload",
    yaxis_title="risk contribution (proxy units)",
    height=620
)
fig.update_xaxes(tickangle=45)
fig.show()



## 10. Export: HTML snapshots and artifacts


In [ ]:

from pathlib import Path

out_dir = Path("./runs") / "dascient_multidim_visual_ops"
out_dir.mkdir(parents=True, exist_ok=True)

panel_path = out_dir / "risk_panel.csv"
panel.to_csv(panel_path, index=False)

s_export = dfm.sample(n=min(8000, len(dfm)), random_state=7)
fig_a = px.scatter_3d(
    s_export, x="joules_per_request", y="co2e_g_per_request", z="slo_risk",
    color="throttling_rate", size="facility_kw",
    hover_data=["site","workload","pue","p95_ms","error_rate","power_capping_rate","telemetry_coverage"],
    opacity=0.65,
    title="3D risk space"
)
fig_a.write_html(out_dir / "fig_3d_risk_space.html", include_plotlyjs="cdn")

temp = np.linspace(-5, 45, 90)
rh = np.linspace(10, 100, 90)
T, RH = np.meshgrid(temp, rh)
cool = np.maximum((T - 18)/22, 0.0)
humid = np.maximum((RH - 60)/40, 0.0)
Z = 1.0 + 0.75*(cool**1.6) + 0.35*(humid**1.4) + 0.22*(cool*humid)
fig_b = go.Figure(data=[go.Surface(x=T, y=RH, z=Z, colorscale="Viridis")])
fig_b.update_layout(title="Cooling amplification surface", height=640)
fig_b.write_html(out_dir / "fig_surface_cooling.html", include_plotlyjs="cdn")

pc = dfm.sample(n=min(6000, len(dfm)), random_state=9)
fig_c = px.parallel_coordinates(
    pc,
    dimensions=["pue","throttling_rate","power_capping_rate","telemetry_coverage","joules_per_request","co2e_g_per_request","slo_risk"],
    color="slo_risk",
    color_continuous_scale="Turbo",
    title="Parallel coordinates"
)
fig_c.write_html(out_dir / "fig_parallel_coordinates.html", include_plotlyjs="cdn")

artifact = {
    "run_ts_utc": RUN_TS,
    "n_rows": int(len(dfm)),
    "panel_csv": str(panel_path),
    "exports": [
        "fig_3d_risk_space.html",
        "fig_surface_cooling.html",
        "fig_parallel_coordinates.html",
    ],
}
with open(out_dir / "run_artifact.json", "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, sort_keys=True)

str(out_dir), artifact



## Closing

Use this package as a visualization spine:

- rotate through multidimensional risk
- animate time evolution
- fly through risk space for narrative clarity
- export interactive HTML for sharing

DaScient
